### Step 1: Download the raw corpus

The cell below downloads the four raw files that make up the Cornell
Movie-Dialogs Corpus (dialogue lines, character metadata, movie metadata,
and conversation structure). The files are pulled from a public GitHub
mirror rather than the original Cornell host, since the original host
occasionally throttles or blocks scripted downloads.

In [6]:
from pathlib import Path
import requests

RAW_DIR = Path("data/raw")
MIRROR_BASE = (
    "https://raw.githubusercontent.com/SudharshanShanmugasundaram/Chatbot/"
    "master/data/cornell%20movie-dialogs%20corpus"
)
FILES = [
    "movie_lines.txt",
    "movie_characters_metadata.txt",
    "movie_titles_metadata.txt",
    "movie_conversations.txt",
]

RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"Downloading Cornell Movie-Dialogs Corpus into {RAW_DIR} ...")
for filename in FILES:
    url = f"{MIRROR_BASE}/{filename}"
    dest = RAW_DIR / filename
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print(f"  {filename}: {len(resp.content):,} bytes -> {dest}")
print("Done.")

  movie_lines.txt: 34,641,919 bytes -> data/raw/movie_lines.txt
  movie_characters_metadata.txt: 705,695 bytes -> data/raw/movie_characters_metadata.txt
  movie_titles_metadata.txt: 67,289 bytes -> data/raw/movie_titles_metadata.txt
  movie_conversations.txt: 6,760,930 bytes -> data/raw/movie_conversations.txt
Done.


### Step 2: Parse and clean the raw corpus

The raw files downloaded above are not directly usable: they are
delimiter-separated text files with inconsistent encoding and occasional
malformed rows. The cell below parses `movie_lines.txt` (every line of
dialogue in the corpus) and `movie_characters_metadata.txt` (character and
movie information), joins them into a single table, and removes empty or
duplicate rows. The result is written to `data/processed/lines.csv`, a
clean table with one row per line of dialogue, including which character
spoke it and in which movie.

In [7]:
import pandas as pd

SEP = " +++$+++ "

def read_raw_lines(path):
    for encoding in ("utf-8", "latin-1"):
        try:
            return path.read_text(encoding=encoding).splitlines()
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"Could not decode {path}")

def load_movie_lines():
    rows = []
    n_malformed = 0
    for raw in read_raw_lines(RAW_DIR / "movie_lines.txt"):
        fields = raw.split(SEP)
        if len(fields) < 5:
            n_malformed += 1
            continue
        line_id, char_id, movie_id, char_name, text = fields[0], fields[1], fields[2], fields[3], SEP.join(fields[4:])
        rows.append({
            "line_id": line_id,
            "character_id": char_id,
            "movie_id": movie_id,
            "character_name": char_name.strip(),
            "text": text.strip(),
        })
    if n_malformed:
        print(f"  Skipped {n_malformed} malformed rows in movie_lines.txt")
    return pd.DataFrame(rows)

def load_character_metadata():
    rows = []
    for raw in read_raw_lines(RAW_DIR / "movie_characters_metadata.txt"):
        fields = raw.split(SEP)
        if len(fields) < 6:
            continue
        rows.append({
            "character_id": fields[0],
            "character_name": fields[1].strip(),
            "movie_id": fields[2],
            "movie_title": fields[3].strip(),
            "gender": fields[4].strip(),
            "credit_position": fields[5].strip(),
        })
    return pd.DataFrame(rows)

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Parsing movie_lines.txt ...")
lines_df = load_movie_lines()
print(f"  {len(lines_df):,} utterances parsed")

print("Parsing movie_characters_metadata.txt ...")
chars_df = load_character_metadata()
print(f"  {len(chars_df):,} characters parsed")

merged = lines_df.merge(chars_df[["character_id", "movie_title"]], on="character_id", how="left")
merged = merged[merged["text"].str.len() > 0].drop_duplicates(subset=["character_id", "text"])

out_path = PROCESSED_DIR / "lines.csv"
merged.to_csv(out_path, index=False)
print(f"Wrote {len(merged):,} clean rows -> {out_path}")

Parsing movie_lines.txt ...
  Skipped 91 malformed rows in movie_lines.txt
  304,713 utterances parsed
Parsing movie_characters_metadata.txt ...
  9,035 characters parsed
Wrote 298,833 clean rows -> data/processed/lines.csv


### Step 3: Save the selected character list

The five characters for this project were finalized during Phase 1:
Jack (Fight Club), Bateman (American Psycho), Alvy (Annie Hall), Ben (The Graduate), and
Erin (Erin Brockovich). Each was selected for having a distinctive speaking
style and a sufficient number of dialogue lines to support fine-tuning.
The cell below writes this selection to
`config/personas.json`, matching the file provided in
Phase 1, so that downstream steps can filter the full corpus down to just
these five characters.

In [8]:
import json

selected_characters = [
    {"character_id": "u5262", "character_name": "JACK", "movie_title": "fight club", "line_count": 354, "persona_tag": "jack"},
    {"character_id": "u327", "character_name": "BATEMAN", "movie_title": "american psycho", "line_count": 335, "persona_tag": "bateman"},
    {"character_id": "u3681", "character_name": "ALVY", "movie_title": "annie hall", "line_count": 459, "persona_tag": "alvy"},
    {"character_id": "u1169", "character_name": "BEN", "movie_title": "the graduate", "line_count": 432, "persona_tag": "ben"},
    {"character_id": "u5076", "character_name": "ERIN", "movie_title": "erin brockovich", "line_count": 338, "persona_tag": "erin"},
]

CONFIG_DIR = Path("config")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
with open(CONFIG_DIR / "personas.json", "w") as f:
    json.dump(selected_characters, f, indent=2)

print(f"Saved {len(selected_characters)} characters")

Saved 5 characters


### Step 4: Filter the corpus to the five selected characters

This step filters `lines.csv` down to only the lines spoken by the five
selected characters, using `config/personas.json` to identify them by
character ID. The result is saved as `data/processed/filtered_lines.csv`.

In [9]:
char_ids = {c["character_id"] for c in selected_characters}
tag_lookup = {c["character_id"]: c["persona_tag"] for c in selected_characters}

lines_df = pd.read_csv(PROCESSED_DIR / "lines.csv")
filtered = lines_df[lines_df["character_id"].isin(char_ids)].copy()
filtered["persona_tag"] = filtered["character_id"].map(tag_lookup)

print("Lines per character:")
print(filtered.groupby("persona_tag").size())

filtered.to_csv(PROCESSED_DIR / "filtered_lines.csv", index=False)
print(f"\nWrote {len(filtered):,} rows -> {PROCESSED_DIR / 'filtered_lines.csv'}")

Lines per character:
persona_tag
alvy       459
bateman    335
ben        432
erin       338
jack       354
dtype: int64

Wrote 1,918 rows -> data/processed/filtered_lines.csv
